In [110]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer

In [111]:
X,y = load_breast_cancer(return_X_y = True, as_frame= True)

In [112]:
print(X.shape,y.shape)

(569, 30) (569,)


In [113]:
X.sample()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
533,20.47,20.67,134.7,1299.0,0.09156,0.1313,0.1523,0.1015,0.2166,0.05419,...,23.23,27.15,152.0,1645.0,0.1097,0.2534,0.3092,0.1613,0.322,0.06386


In [114]:
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, make_scorer, recall_score

In [115]:
X_train,X_test,y_train,y_test = train_test_split(X,y,stratify=y,shuffle=True,test_size=0.2,random_state=42)

In [116]:
print(X_train.shape,X_test.shape,y_train.shape,y_test.shape)

(455, 30) (114, 30) (455,) (114,)


In [117]:
cv = StratifiedKFold(n_splits = 5, shuffle = True,random_state=42)
scoring = make_scorer(recall_score,pos_label=0)

In [118]:
# baseline classifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_val_score


dummy = DummyClassifier(strategy='most_frequent')
print(cross_val_score(dummy,X_train,y_train,cv = cv, scoring = 'accuracy').mean())

0.6263736263736264


In [119]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier




models = {
    "logreg": make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000)),
    "svm": make_pipeline(StandardScaler(),SVC()),
    "rf": RandomForestClassifier(random_state=42)
}


In [120]:
for name, model in models.items():
    scores = cross_val_score(model, X_train,y_train,cv=cv,scoring=scoring)
    print(f"{name}: {scores.mean():.3f} +- {scores.std():.3f}")

logreg: 0.965 +- 0.029
svm: 0.953 +- 0.040
rf: 0.959 +- 0.030


In [121]:
from sklearn.model_selection import GridSearchCV

params =  {
    'logisticregression__C': [0.01,0.02,0.1,1,2,10,100],
    'logisticregression__penalty': ["l2"],
    'logisticregression__class_weight': [None, 'balanced']
}


grid = GridSearchCV(models["logreg"],param_grid=params,cv = cv,scoring = scoring,n_jobs=1)
grid.fit(X_train,y_train)
print(grid.best_params_,grid.best_score_)


/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalt

{'logisticregression__C': 1, 'logisticregression__class_weight': 'balanced', 'logisticregression__penalty': 'l2'} 0.9764705882352942


/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalt

In [122]:
print(grid.best_params_,grid.best_score_)

{'logisticregression__C': 1, 'logisticregression__class_weight': 'balanced', 'logisticregression__penalty': 'l2'} 0.9764705882352942


In [123]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report

best = grid.best_estimator_
y_cv_pred = cross_val_predict(best, X_train, y_train, cv=cv)

print(confusion_matrix(y_train, y_cv_pred))
print(classification_report(y_train, y_cv_pred, target_names=["malignant", "benign"]))


[[166   4]
 [  7 278]]
              precision    recall  f1-score   support

   malignant       0.96      0.98      0.97       170
      benign       0.99      0.98      0.98       285

    accuracy                           0.98       455
   macro avg       0.97      0.98      0.97       455
weighted avg       0.98      0.98      0.98       455



/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalt

In [125]:
import numpy as np
from sklearn.metrics import recall_score, precision_score

proba_malig = cross_val_predict(best, X_train, y_train, cv=cv, method="predict_proba")[:, 0]

for t in [0.5, 0.4, 0.3, 0.2, 0.1]:
    pred_malig = proba_malig >= t
    y_pred_t = np.where(pred_malig, 0, 1)
    print(t,
          "recall:", recall_score(y_train, y_pred_t, pos_label=0),
          "precision:", precision_score(y_train, y_pred_t, pos_label=0))


0.5 recall: 0.9764705882352941 precision: 0.9595375722543352
0.4 recall: 0.9823529411764705 precision: 0.943502824858757
0.3 recall: 0.9823529411764705 precision: 0.9226519337016574
0.2 recall: 0.9882352941176471 precision: 0.8795811518324608
0.1 recall: 0.9941176470588236 precision: 0.8325123152709359


/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalt

In [126]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

threshold = 0.4   # chosen from train CV, not from the test set

proba_malig_test = best.predict_proba(X_test)[:, 0]   # column 0 = P(malignant)
y_pred = np.where(proba_malig_test >= threshold, 0, 1)

print(classification_report(y_test, y_pred, target_names=["malignant", "benign"]))
print(confusion_matrix(y_test, y_pred))


              precision    recall  f1-score   support

   malignant       0.89      0.98      0.93        42
      benign       0.99      0.93      0.96        72

    accuracy                           0.95       114
   macro avg       0.94      0.95      0.94       114
weighted avg       0.95      0.95      0.95       114

[[41  1]
 [ 5 67]]


In [127]:
import joblib
joblib.dump({"model": best, "threshold": 0.4}, "breast_cancer_model.joblib")


['breast_cancer_model.joblib']